In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path

proje = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()

In [ ]:
dosya_yollari = [
    proje / "data" / "deneme1_girdi.csv",
    proje.parent / "data" / "deneme1_girdi.csv",
    proje / "data" / "processed" / "deneme1.csv",
    proje.parent / "data" / "processed" / "deneme1.csv",
]

dosya = next((yol for yol in dosya_yollari if yol.is_file()), None)
if dosya is None:
    raise FileNotFoundError("deneme1_girdi.csv bulunamadi. data klasorunun konumunu kontrol et.")

df = pd.read_csv(dosya)
print("Veri dosyasi:", dosya)

In [ ]:
y = df["price"]
X = df.drop(columns=["price", "price_per_sqm", "listing_id", "total_rooms", "orientation"])

In [ ]:
X_train = X.sample(frac=0.70, random_state=42).copy()
X_kalan = X.drop(index=X_train.index)

X_val = X_kalan.sample(frac=0.50, random_state=42).copy()
X_test = X_kalan.drop(index=X_val.index).copy()

y_train = y.loc[X_train.index]
y_val = y.loc[X_val.index]
y_test = y.loc[X_test.index]

In [ ]:
sutunlar = ["bathroom_count", "total_floors"]

medyanlar = X_train[sutunlar].median()

X_train[sutunlar] = X_train[sutunlar].fillna(medyanlar)
X_val[sutunlar] = X_val[sutunlar].fillna(medyanlar)
X_test[sutunlar] = X_test[sutunlar].fillna(medyanlar)

In [ ]:
for veri in [X_train, X_val, X_test]:
    veri["exchange"] = (veri["exchange"] == "Yes").astype(int)
    veri["credit_eligible"] = (veri["credit_eligible"] == "Eligible").astype(int)
    veri["furnished"] = (veri["furnished"] == "Furnished").astype(int)
    veri["is_in_complex"] = veri["is_in_complex"].astype(int)
    veri["is_top_floor"] = veri["is_top_floor"].fillna(False).astype(int)
    veri["is_ground_floor"] = veri["is_ground_floor"].fillna(False).astype(int)

In [ ]:
for veri in [X_train, X_val, X_test]:
    veri["usage_status"] = veri["usage_status"].replace("Not Specified", pd.NA)

kategorikler = ["district", "neighborhood", "heating_type", "usage_status", "deed_status"]

X_train = pd.get_dummies(X_train, columns=kategorikler, dtype=int)
X_val = pd.get_dummies(X_val, columns=kategorikler, dtype=int)
X_test = pd.get_dummies(X_test, columns=kategorikler, dtype=int)

X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [ ]:
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

In [ ]:
print("Train eksik:", X_train.isna().sum().sum())
print("Validation eksik:", X_val.isna().sum().sum())
print("Test eksik:", X_test.isna().sum().sum())

## 1. Sayisal verileri normalize etme

In [ ]:
sayisal = ["rooms", "halls", "gross_sqm", "net_sqm", "total_floors", "building_age", "bathroom_count"]

ortalama = X_train[sayisal].mean()
standart_sapma = X_train[sayisal].std().replace(0, 1)

for veri in [X_train, X_val, X_test]:
    veri[sayisal] = (veri[sayisal] - ortalama) / standart_sapma

print("Train eksik:", X_train.isna().sum().sum())
print("Validation eksik:", X_val.isna().sum().sum())
print("Test eksik:", X_test.isna().sum().sum())

## 2. TensorFlow icin verileri hazirlama

In [ ]:
tf.random.set_seed(42)
np.random.seed(42)

X_train_tf = tf.constant(X_train.to_numpy(dtype=np.float32))
X_val_tf = tf.constant(X_val.to_numpy(dtype=np.float32))
X_test_tf = tf.constant(X_test.to_numpy(dtype=np.float32))

y_train_tf = tf.constant(y_train.to_numpy(dtype=np.float32).reshape(-1, 1) / 1_000_000)
y_val_tf = tf.constant(y_val.to_numpy(dtype=np.float32).reshape(-1, 1) / 1_000_000)

print("Giris ozellik sayisi:", X_train_tf.shape[1])
print("Train / Validation / Test:", len(y_train), len(y_val), len(y_test))

## 3. Uc ANN mimarisi

Ayni train ve validation verileriyle 100-50, 50-25 ve 150-100 modellerini karsilastiriyoruz. Butun gizli katmanlarda ReLU, cikista dogrusal aktivasyon var.

In [ ]:
mimariler = [(100, 50), (50, 25), (150, 100)]

modeller_klasoru = proje / "models"
sonuclar_klasoru = proje / "results"
modeller_klasoru.mkdir(parents=True, exist_ok=True)
sonuclar_klasoru.mkdir(parents=True, exist_ok=True)

## 4. TensorFlow ile ANN egitimi

Keras kullanmadan tf.Variable, tf.GradientTape ve Adam ile egitim yapiyoruz. Her modelde en dusuk validation MSE degerine ulasan agirliklar saklaniyor. Maksimum 150 epoch, patience 15, batch size 64.

In [ ]:
def model_olustur(birinci, ikinci):
    giris = int(X_train_tf.shape[1])

    w1 = tf.Variable(tf.random.normal([giris, birinci], stddev=(2.0 / giris) ** 0.5))
    b1 = tf.Variable(tf.zeros([birinci]))

    w2 = tf.Variable(tf.random.normal([birinci, ikinci], stddev=(2.0 / birinci) ** 0.5))
    b2 = tf.Variable(tf.zeros([ikinci]))

    w3 = tf.Variable(tf.random.normal([ikinci, 1], stddev=(2.0 / ikinci) ** 0.5))
    b3 = tf.Variable(tf.zeros([1]))

    return [w1, b1, w2, b2, w3, b3]


def tahmin(x, parametreler):
    w1, b1, w2, b2, w3, b3 = parametreler
    x = tf.nn.relu(tf.matmul(x, w1) + b1)
    x = tf.nn.relu(tf.matmul(x, w2) + b2)
    return tf.matmul(x, w3) + b3

In [ ]:
def modeli_egit(birinci, ikinci):
    isim = f"ann_{birinci}_{ikinci}"
    tf.random.set_seed(42)
    parametreler = model_olustur(birinci, ikinci)
    optimizer = tf.optimizers.Adam(learning_rate=0.001)

    train_verisi = tf.data.Dataset.from_tensor_slices((X_train_tf, y_train_tf))
    train_verisi = train_verisi.shuffle(len(y_train), seed=42).batch(64)

    @tf.function
    def egitim_adimi(x_batch, y_batch):
        with tf.GradientTape() as tape:
            y_tahmin = tahmin(x_batch, parametreler)
            loss = tf.reduce_mean(tf.square(y_batch - y_tahmin))
        gradyanlar = tape.gradient(loss, parametreler)
        optimizer.apply_gradients(zip(gradyanlar, parametreler))

    train_gecmisi = []
    val_gecmisi = []
    en_iyi_val_loss = float("inf")
    en_iyi_agirliklar = None
    en_iyi_epoch = 0
    bekleme = 0

    for epoch in range(1, 151):
        for x_batch, y_batch in train_verisi:
            egitim_adimi(x_batch, y_batch)

        train_loss = float(tf.reduce_mean(tf.square(y_train_tf - tahmin(X_train_tf, parametreler))).numpy())
        val_loss = float(tf.reduce_mean(tf.square(y_val_tf - tahmin(X_val_tf, parametreler))).numpy())
        train_gecmisi.append(train_loss)
        val_gecmisi.append(val_loss)

        if val_loss < en_iyi_val_loss:
            en_iyi_val_loss = val_loss
            en_iyi_agirliklar = [p.numpy().copy() for p in parametreler]
            en_iyi_epoch = epoch
            bekleme = 0
        else:
            bekleme += 1

        if epoch == 1 or epoch % 10 == 0:
            print(f"{isim} | Epoch {epoch:3d} | Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f}")

        if bekleme >= 15:
            print(f"{isim}: validation hatasi iyilesmedigi icin egitim durdu.")
            break

    if en_iyi_agirliklar is None:
        raise ValueError(f"{isim} icin gecerli bir validation loss hesaplanamadi.")

    for parametre, agirlik in zip(parametreler, en_iyi_agirliklar):
        parametre.assign(agirlik)

    gercek = y_val.to_numpy(dtype=np.float64)
    tahmini = tahmin(X_val_tf, parametreler).numpy().reshape(-1).astype(np.float64) * 1_000_000

    mae = np.mean(np.abs(gercek - tahmini))
    rmse = np.sqrt(np.mean((gercek - tahmini) ** 2))
    r2 = 1 - np.sum((gercek - tahmini) ** 2) / np.sum((gercek - gercek.mean()) ** 2)
    mape = np.mean(np.abs((gercek - tahmini) / np.maximum(np.abs(gercek), 1))) * 100

    klasor = modeller_klasoru / isim
    klasor.mkdir(parents=True, exist_ok=True)

    checkpoint = tf.train.Checkpoint(w1=parametreler[0], b1=parametreler[1],
                                     w2=parametreler[2], b2=parametreler[3],
                                     w3=parametreler[4], b3=parametreler[5])
    kayit = checkpoint.save(str(klasor / "ann"))

    np.savez(
        klasor / "onisleme.npz",
        sutunlar=X_train.columns.to_numpy(),
        sayisal_sutunlar=np.array(sayisal),
        ortalama=ortalama.to_numpy(),
        standart_sapma=standart_sapma.to_numpy(),
        medyan_sutunlar=medyanlar.index.to_numpy(),
        medyanlar=medyanlar.to_numpy(),
        fiyat_olcegi=1_000_000,
    )

    print(f"{isim} | En iyi epoch: {en_iyi_epoch} | MAE: {mae:,.0f} TL | RMSE: {rmse:,.0f} TL | R2: {r2:.4f}")
    print("Kaydedildi:", kayit)

    sonuc = {
        "model": isim,
        "katman_1": birinci,
        "katman_2": ikinci,
        "en_iyi_epoch": en_iyi_epoch,
        "val_mse_milyon_tl_kare": en_iyi_val_loss,
        "val_mae_tl": mae,
        "val_rmse_tl": rmse,
        "val_r2": r2,
        "val_mape_yuzde": mape,
    }
    return sonuc, train_gecmisi, val_gecmisi

## 5. Uc modeli egitme ve ayri kaydetme

Model kayitlari `models/ann_100_50/`, `models/ann_50_25/` ve `models/ann_150_100/` klasorlerine gider. Eski `models/ann_100_50-1` kaydina dokunulmaz.

In [ ]:
sonuclar = []
gecmisler = {}

for birinci, ikinci in mimariler:
    sonuc, train_gecmisi, val_gecmisi = modeli_egit(birinci, ikinci)
    sonuclar.append(sonuc)
    gecmisler[sonuc["model"]] = (train_gecmisi, val_gecmisi)

karsilastirma = pd.DataFrame(sonuclar).sort_values("val_mae_tl").reset_index(drop=True)
karsilastirma.to_csv(sonuclar_klasoru / "model_karsilastirma.csv", index=False)

karsilastirma

## 6. Validation karsilastirmasi

Dusuk MAE ve RMSE, yuksek R2 daha iyidir. Modellere ayni validation verisini verdigimiz icin adil bir karsilastirma yapiyoruz.

In [ ]:
print(karsilastirma[["model", "en_iyi_epoch", "val_mae_tl", "val_rmse_tl", "val_r2", "val_mape_yuzde"]].to_string(index=False))

plt.figure(figsize=(9, 5))
for isim, (train_gecmisi, val_gecmisi) in gecmisler.items():
    plt.plot(val_gecmisi, label=isim)
plt.xlabel("Epoch")
plt.ylabel("Validation MSE (milyon TL)^2")
plt.title("Uc ANN modelinin validation kaybi")
plt.legend()
plt.show()

## 7. En iyi modelin test sonucunu hesaplama

En dusuk validation MAE'ye sahip modeli secip test verisini **yalnizca bir kez** degerlendiriyoruz. Test verisi egitimde veya model seciminde kullanilmadi.

In [ ]:
en_iyi = karsilastirma.iloc[0]
en_iyi_isim = en_iyi["model"]
print("Validation MAE'ye gore secilen model:", en_iyi_isim)

parametreler = model_olustur(int(en_iyi["katman_1"]), int(en_iyi["katman_2"]))
checkpoint = tf.train.Checkpoint(w1=parametreler[0], b1=parametreler[1],
                                 w2=parametreler[2], b2=parametreler[3],
                                 w3=parametreler[4], b3=parametreler[5])
kayit = tf.train.latest_checkpoint(str(modeller_klasoru / en_iyi_isim))
if kayit is None:
    raise FileNotFoundError("Secilen modelin checkpoint dosyasi bulunamadi.")
checkpoint.restore(kayit).assert_existing_objects_matched()

gercek = y_test.to_numpy(dtype=np.float64)
tahmini = tahmin(X_test_tf, parametreler).numpy().reshape(-1).astype(np.float64) * 1_000_000

test_mae = np.mean(np.abs(gercek - tahmini))
test_rmse = np.sqrt(np.mean((gercek - tahmini) ** 2))
test_r2 = 1 - np.sum((gercek - tahmini) ** 2) / np.sum((gercek - gercek.mean()) ** 2)
test_mape = np.mean(np.abs((gercek - tahmini) / np.maximum(np.abs(gercek), 1))) * 100

print(f"Test MAE:  {test_mae:,.0f} TL")
print(f"Test RMSE: {test_rmse:,.0f} TL")
print(f"Test R2:   {test_r2:.4f}")
print(f"Test MAPE: %{test_mape:.2f}")

pd.DataFrame([{
    "model": en_iyi_isim,
    "test_mae_tl": test_mae,
    "test_rmse_tl": test_rmse,
    "test_r2": test_r2,
    "test_mape_yuzde": test_mape,
}]).to_csv(sonuclar_klasoru / "test_sonucu.csv", index=False)

print("Tum modeller:", modeller_klasoru)
print("Deney sonuclari:", sonuclar_klasoru)